### **nef_translocation - PART 2 - From segmentation to measurement**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2025/11/07

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
import datetime
import os
import numpy as np
import pandas as pd
import tifffile
from skimage.measure import regionprops_table
# from image_preparation.preprocess_image_and_label import preprocess_image_and_label, parallel_preprocess_image_and_label
from image_preparation.preprocess_image_and_label import preprocess_image_and_label
from image_preparation.make_imagej_metadata import imagej_compatible_metadata_dict
from utils.save_image import tifffile_save_ometiff
# from feature_extraction.extract_features import extract_features, replace_column_target_measurement_channel_pair
# from feature_extraction.add_pvalue import add_pvalue_column


### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modified are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [2]:
# indicate the path to the directory storing the input images
fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\nef_translocation\develop\251105_multi_label_error\fov"

# indicate the path to the directory storing the segmentations
segmentation_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\nef_translocation\develop\251105_multi_label_error\seg"

# indicate the path to the directory where outputs will be saved
output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\nef_translocation\develop\251105_multi_label_error"

# metadata_df directory - indicate the full path to the directory where part1 metadata have been saved
metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\nef_translocation\develop\251105_multi_label_error\metadata\proc_file_info"

# metadata file name
metadata_file = "251105_nef_translocation_metadata.csv"

# indicate if nucleus and cell segmentation, after filtering, should be saved
save_nuc_cell_segmentation = True


# --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---
# indicate the original dtype of the input images - this is used to calculate the min and max pixels intensity values
# if pixels have such values, they are not used for quantification.
original_dtype = np.uint16

# indicate the channel axis position - image preprocessing is iterated over the individual channels
channel_axis = 0

# indicate the highpass threshold for nuclei area
# nuclei with area smaller than the threshold will be discarder from the analysis
# unit is number of pixel
# it is possible to indicate None - when None is indicated, no filter is used
nuc_area_highpass=500 # recommended to use 500

# indicate the lowpass threshold for nuclei area
# nuclei with area bigger than the threshold will be discarder from the analysis
# unit is number of pixel
# it is possible to indicate None - when None is indicated, no filter is used
nuc_area_lowpass=50000 # recommended to use 50000

# indicate the highpass threshold for cell area
# cells with area smaller than the threshold will be discarder from the analysis
# unit is number of pixel
# it is possible to indicate None - when None is indicated, no filter is used
cel_area_highpass=1000 # recommended to use 2000

# indicate the lowpass threshold for cell area
# cells with area bigger than the threshold will be discarder from the analysis
# unit is number of pixel
# it is possible to indicate None - when None is indicated, no filter is used
cel_area_lowpass=100000 # recommended to use 200000

# initialize background function as None - NOTE: it is possible to pass here an array to be used for subtracting the background
# and correct for uneven illumination
# when this parameter is set to None, no local background correction is made
background_function = None

# indicate background dtype - if a background function is passed (see above) it is important to convert the data type of the
# field of view, for allocating negative numbers. The present parameter specifies the datatype to be used for this operation
background_dtype = np.int32

# # rolling ball radius - used for background estimation with the rolling ball algorithm - this is not used
# rolling_ball_radius = 200

# # dask arguments - this is not used
# use_dask = True
# dask_scheduler = 'distributed' # 'synchronous', 'threads', 'processes', 'distributed'

# sigma for gaussian smoothing
sigma = 0.6

# gaussian smoothin kwargs
gaussian_kwargs = {'preserve_range':True}

# separator - dont modify the following line, this is just used to avoid any variable in the following cell to be hard coded
separator = '_'

# ome suffixb - used to save ome.tif files
ome_suffix = ".ome.tif"

# cytosol suffix - the suffix added to the cytosol segmentation masks
cytosol_suffix = "Ct"

# nucleus suffix - the suffix added to the nucleus segmentation masks
nucleus_suffix = "Nc"

# suffix to add to nucleus segmentation mask if saved after filtering
filter_nuc_suffix = "F"

# suffix to add to cell segmentation mask if saved after filtering
filter_cel_suffix = "F"

# istantiate the secondary output directory - secondary output directory is used to save the hyperparameters of the notebook
# normally, if the notebook is run after part1, the following line should not be changed
secondary_output_directory = os.path.join(os.getcwd(), "secondary_output")



### Open the metadata dataframe

Run the following cell.

Don't modify the following cell.

In [3]:
# open metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory,metadata_file))

# copy metadata file
metadata_df = metadata_df_i.copy()

# drop columns which are not needed
metadata_df.drop(['Unnamed: 0'], axis=1, inplace=True)

# # visualize metadata file
metadata_df.head()
# metadata_df


,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,donor,transfection,stiffness,...,size_c,size_z,size_y,physical_size_y,size_x,physical_size_x,dims_order,segmentation_date_yymmdd_Nc_Cl,cell_segmentation,nucleus_segmentation
0,D26_GFP_Glass_CD3CD28_5min_.nd2,A1,251105,D26_GFP_Glass_CD3CD28_5min__s0.ome.tif,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CrestOptics X-Light V3 - spinning ...,"Plan Apo Lambda D 100x Oil/1,45/0,13",D26,GFP,Glass,...,5,1,2720,0.065,2720,0.065,CYX,251105,D26_GFP_Glass_CD3CD28_5min__s0_Cl.ome.tif,D26_GFP_Glass_CD3CD28_5min__s0_Nc.ome.tif
1,D26_GFP_Glass_CD3CD28_5min_.nd2,A2,251105,D26_GFP_Glass_CD3CD28_5min__s1.ome.tif,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CrestOptics X-Light V3 - spinning ...,"Plan Apo Lambda D 100x Oil/1,45/0,13",D26,GFP,Glass,...,5,1,2720,0.065,2720,0.065,CYX,251105,D26_GFP_Glass_CD3CD28_5min__s1_Cl.ome.tif,D26_GFP_Glass_CD3CD28_5min__s1_Nc.ome.tif
2,D26_GFP_Glass_CD3CD28_5min_.nd2,A3,251105,D26_GFP_Glass_CD3CD28_5min__s2.ome.tif,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CrestOptics X-Light V3 - spinning ...,"Plan Apo Lambda D 100x Oil/1,45/0,13",D26,GFP,Glass,...,5,1,2720,0.065,2720,0.065,CYX,251105,D26_GFP_Glass_CD3CD28_5min__s2_Cl.ome.tif,D26_GFP_Glass_CD3CD28_5min__s2_Nc.ome.tif
3,D26_GFP_Glass_CD3CD28_5min_.nd2,B3,251105,D26_GFP_Glass_CD3CD28_5min__s3.ome.tif,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CrestOptics X-Light V3 - spinning ...,"Plan Apo Lambda D 100x Oil/1,45/0,13",D26,GFP,Glass,...,5,1,2720,0.065,2720,0.065,CYX,251105,D26_GFP_Glass_CD3CD28_5min__s3_Cl.ome.tif,D26_GFP_Glass_CD3CD28_5min__s3_Nc.ome.tif
4,D26_GFP_Glass_CD3CD28_5min_.nd2,B2,251105,D26_GFP_Glass_CD3CD28_5min__s4.ome.tif,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CrestOptics X-Light V3 - spinning ...,"Plan Apo Lambda D 100x Oil/1,45/0,13",D26,GFP,Glass,...,5,1,2720,0.065,2720,0.065,CYX,251105,D26_GFP_Glass_CD3CD28_5min__s4_Cl.ome.tif,D26_GFP_Glass_CD3CD28_5min__s4_Nc.ome.tif


### Create a background function for each channel - to correct the local background for uneven illumination

This has been evaluated and it was decided that it is not required.

For this reason, the cell below is commented out.

It is possible to uncomment it and run it

In [9]:
# import napari

# # get all field of view names as a list
# fields_of_view = list(metadata_df['ome_tif_file_name'].to_numpy())

# # Initialize a list to store fields of view
# fov_list = []

# # iterate over all fields of view, open the, and add them to the list
# for fov in fields_of_view:
    
#     # process file only if file exists
#     if os.path.exists(os.path.join(fov_directory, str(fov))):

#         # open the field of view
#         original_fov = tifffile.imread(os.path.join(fov_directory, fov))

#         # append the field of view to the list
#         fov_list.append(original_fov)
#         print(original_fov.shape)

# # stack all fields of view in a numpy array
# fov_array = np.stack(fov_list, axis=0)
# print(fov_array.shape)

# # get an average projection of all fields of view per each channel
# background_function = np.mean(fov_array, axis=0)
# (background_function.shape)

# viewer = napari.Viewer()
# viewer.add_image(background_function, channel_axis=0)


### MAIN LOOP
#### 3.1. Preprocess image and segmentation mask
#### 3.2. Extract features
#### 3.3. Save results 

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [ ]:

# initialize a global collection list
glob_features_collection_list = []


# iterate through the rowa of the metadata dataframe
for i in metadata_df.index:

    print("---------    ---------")
    
    # ---------
    # OPEN FIELD OF VIEW AND SEGMENTATION
    # ---------

    # get the field of view file
    field_of_view_file = metadata_df.loc[i, 'ome_tif_file_name']

    # process file only if file exists
    if os.path.exists(os.path.join(fov_directory, str(field_of_view_file))):
        
        print(f"working on {field_of_view_file}")

        # open the field of view
        field_of_view_i = tifffile.imread(os.path.join(fov_directory, field_of_view_file))

#         # get the cytosol segmentation file - this is not anymore from part 1 notebook, but it is generated below
#         cytosol_segmentation_file = metadata_df.loc[i, 'cytosol_segmentation']
#         cytosol_segmentation = tifffile.imread(os.path.join(segmentation_directory, cytosol_segmentation_file))

        # get the nucleus segmentation file and open the segmentation mask
        nucleus_segmentation_file = metadata_df.loc[i, 'nucleus_segmentation']
        nucleus_segmentation_i = tifffile.imread(os.path.join(segmentation_directory, nucleus_segmentation_file))
        
        # get the cell segmentation file and open the segmentation mask
        cell_segmentation_file = metadata_df.loc[i, 'cell_segmentation']
        cell_segmentation_i = tifffile.imread(os.path.join(segmentation_directory, cell_segmentation_file))
        
        # ---------
        # PREPROCESSING CELL AND NUCLEUS MASKS - OBTAIN THE CYTOSOL SEGMENTATION MASK FROM THE PREPROCESSING RESULT

        # 1) Channel axis is moved to position -1, for compatibility with skimage.measure.regionprops
        # 2) Cells and nuclei with abnormal dimension (smaller than highpass_threshold or bigger than lowpass_threshold) are removed
        # 3) Cells with no nuclei and nuclei with no cells are removed
        # 4) Cells touching the image edges are removed
        # 5) Nuclei are forced to be entirely contained into corresponding cell masks
        # 6) Nuclei belonging to same cell are joined and assigned the same label value of the "mother" cell
        # 7) Pixels having the min or max values of original image data type in any of the channels are discarded
        # 8) The matching of labels between nuclei and corresponding cells is double-checked
        # 9) Subtract preprocessed nucleus mask from preprocessed cell mask to obtain cytosol segmentation mask - NOTE: the label matchin is ensured during this process
        # 10) Channel are, individually and independently, very mildly gaussian smoothed
        # FINAL NOTE: if a background function is passed (per each channel), the function is substracted from the respective channel, before point 10 (gaussian smoothing)
        # ---------

        field_of_view, nucleus_segmentation, cell_segmentation, cytosol_segmentation = preprocess_image_and_label(field_of_view_i,
                                                                                                                  nucleus_segmentation_i,
                                                                                                                  cell_segmentation_i,
                                                                                                                  nuc_area_highpass=nuc_area_highpass,
                                                                                                                  nuc_area_lowpass=nuc_area_lowpass,
                                                                                                                  cel_area_highpass=cel_area_highpass,
                                                                                                                  cel_area_lowpass=cel_area_lowpass,
                                                                                                                  channel_axis=channel_axis,
                                                                                                                  sigma=sigma,
                                                                                                                  gaussian_kwargs=gaussian_kwargs,
                                                                                                                  original_dtype=original_dtype,
                                                                                                                  background_function=background_function,
                                                                                                                  background_dtype=background_dtype)
        
        # print progress
        print("Image preprocessing is done. Cytosol segmentation mask obtained")
        
        # ---------
        # SAVE CYTOSOL SEGMENTATION RESULTS
        # ---------

        # form cytosol segmentation saving name
        cyt_save_name = f"{field_of_view_file.removesuffix(ome_suffix)}{separator}{cytosol_suffix}{ome_suffix}"

        # form a metadata dictionary to be used for saving metadata in the segmentation masks
        segmentation_metadata_dict = {'processing_date_yymmdd':datetime.datetime.now().strftime('%y%m%d')}
        for clm in metadata_df.columns:
            if clm in ['raw_file_name', 'scene_name', 'ome_tif_file_name', 'location', 'microscope', 'objective', 'donor',
                       'transfection', 'stiffness', 'stimulation', 'time_of_stimulation',
                       'physical_size_unit_x', 'physical_size_unit_y', 'physical_size_y', 'size_x',
                       'physical_size_x', ]:
                
                segmentation_metadata_dict[clm]=metadata_df.loc[i,clm]

        # change metadata dictionary to an ImageJ compatible format
        imagej_segmentation_metadata_dict = imagej_compatible_metadata_dict(segmentation_metadata_dict)

        # save the cytosol segmentation
        tifffile_save_ometiff(os.path.join(segmentation_directory,cyt_save_name),
                                    data=cytosol_segmentation,
                                    imagej=True,
                                    photometric="minisblack",
                                    metadata=imagej_segmentation_metadata_dict)

        print("Cytosol segmentation has been saved")
        
        # save nucleus and cell segmentation if desired
        if save_nuc_cell_segmentation:
            
            # form nucleus and cell segmentation saving names
            nuc_save_name = f"{nucleus_segmentation_file.removesuffix(ome_suffix)}{separator}{filter_nuc_suffix}{ome_suffix}"
            cel_save_name = f"{cell_segmentation_file.removesuffix(ome_suffix)}{separator}{filter_cel_suffix}{ome_suffix}"


            # save the nucleus and segmentation
            tifffile_save_ometiff(os.path.join(segmentation_directory,nuc_save_name),
                                        data=nucleus_segmentation,
                                        imagej=True,
                                        photometric="minisblack",
                                        metadata=imagej_segmentation_metadata_dict)
            
            # save the nucleus and segmentation
            tifffile_save_ometiff(os.path.join(segmentation_directory,cel_save_name),
                                        data=cell_segmentation,
                                        imagej=True,
                                        photometric="minisblack",
                                        metadata=imagej_segmentation_metadata_dict)
            
            print("Nucleus and cell segmentation has been saved - NOTE: THEIR NAMES ARE NOT UPDATED IN THE METADATA DATAFRAME")

        # ---------
        # MEASUREMENT EXTRACTION
        # ---------

        # extract measurements for the nucleus and cytosol segmentations

        nucleus_measurements = pd.DataFrame(regionprops_table(nucleus_segmentation,
                                                              intensity_image=field_of_view,
                                                              properties=['label', 'area', 'centroid', 'intensity_mean', 'intensity_max', 'intensity_min']))

        cytosol_measurements = pd.DataFrame(regionprops_table(cytosol_segmentation,
                                                              intensity_image=field_of_view,
                                                              properties=['label', 'area', 'centroid', 'intensity_mean', 'intensity_max', 'intensity_min']))

        # rename columns to specify nuclei and cytosol measurements
        nucleus_measurements.rename(columns={col:f"{nucleus_suffix}{separator}{col}" for col in nucleus_measurements.columns if col!='label'}, inplace=True)
        cytosol_measurements.rename(columns={col:f"{cytosol_suffix}{separator}{col}" for col in cytosol_measurements.columns if col!='label'}, inplace=True)
                
        # join initial nucleus and cytosol measurements
        fov_glob_features = nucleus_measurements.merge(cytosol_measurements, on='label', how='outer')

        # print progress
        print("measurements: done")

        # ---------
        # CALCULATE BACKGROUND OFFSET PER CHANNEL - THIS IS DONE BY CALCULATING THE MEDIAN OF THE PIXELS NOT BELONGING TO ANY SEGMENTATION MASK
        # ---------

        # join cell and nucleus segmentation masks - NOTE: for this purpose, the original segmentation masks are used, not the preprocessed ones
        # this is because the preprocessed masks have removed some cells/nuclei (e.g. those touching the image edge), in addition, the orignal
        # segmentation have likely picked up artifacts, which are therefore excluded from the background calculation
        
        background_mask = np.where(cell_segmentation_i>0,0,1)
        background_mask = np.where(nucleus_segmentation_i>0,background_mask,1)
        
        # iterate through the channels of the field of view - NOTE: channels are in position -1 from the preprocessing
        for c in range(field_of_view.shape[-1]):
            
            # get the channel array for the field of view
            fov_ch = field_of_view[...,c]

            # calculate the median of the pixels which are not in any segmentation mask
            ch_background_offset = np.median(fov_ch[background_mask>0])
            
            # add ch_backgound_offset to the measurement dataframe for the field of view
            fov_glob_features[f'background_offset-{c}'] = [ch_background_offset for i in range(fov_glob_features.shape[0])]

        # ---------
        # ADD METADATA
        # ---------
        # get field_of_view metadata
        fov_metadata = metadata_df[metadata_df['ome_tif_file_name']==field_of_view_file].copy()
        
        # add cytosol segmentation metadata
        fov_metadata[f'segmentation_date_yymmdd{separator}{cytosol_suffix}'] = datetime.datetime.now().strftime('%y%m%d')
        fov_metadata['cytosol_segmentation'] = cyt_save_name
        
        # transform metadata series into a dataframe with elements as columns, with the same number of rows as the
        # glob_features dataframe and the values repeated over the rows
        fov_metadata_df = pd.DataFrame({k: [fov_metadata[k].to_numpy()[0]] * fov_glob_features.shape[0] for k in fov_metadata})

        # concatenate metadata and glob_features
        fov_glob_features = pd.concat([fov_metadata_df, fov_glob_features], axis=1, ignore_index=False)

        # print progress
        print("add metadata: done")

        # ---------
        # COLLECT MEASUREMENTS IN GLOBAL COLLECTION LIST
        # ---------
        glob_features_collection_list.append(fov_glob_features)

# concatenate measurements of all fields of view
glob_features = pd.concat(glob_features_collection_list, axis=0, ignore_index=True)

print("")
print("=== === ===")
print("feature extraction finished")

# save the result
glob_features_saving_name = f"{datetime.datetime.now().strftime('%y%m%d')}_nef_translocation_glob_measurements.csv"
glob_features.to_csv(os.path.join(output_directory, glob_features_saving_name))

# save results in an excel compatible format
glob_features_saving_name_excl = f"{datetime.datetime.now().strftime('%y%m%d')}_nef_translocation_glob_measurements_excel.csv"
glob_features.to_csv(os.path.join(output_directory, glob_features_saving_name_excl), sep=';', decimal=',', index=False)

print("results saved")


---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------    ---------
---------  

### Save analysis hyperparameters

Run the following cell.

Don't modify the following cell.

In [12]:
# initialize a boolean variable which is True if background_function is not None, False otherwise
if hasattr(background_function, "__len__"):
    background_function_is_used = True
else:
    background_function_is_used = False

# # collect hyperparameters in a dictionary
hyperparameter_dict = {'fov_directory':fov_directory,
                       'segmentation_directory':segmentation_directory,
                       'output_directory':output_directory,
                       'metadata_directory':metadata_directory,
                       'metadata_file':metadata_file,
                       'save_nuc_cell_segmentation':save_nuc_cell_segmentation,
                       'channel_axis':channel_axis,
                       'nuc_area_highpass':nuc_area_highpass,
                       'nuc_area_lowpass':nuc_area_lowpass,
                       'cel_area_highpass':cel_area_highpass,
                       'cel_area_lowpass':cel_area_lowpass,
                       'original_dtype':original_dtype,
                       'background_function_is_used':background_function_is_used,
                       'sigma':sigma,
                       'gaassian_kwargs':gaussian_kwargs,
                       'background_dtype':background_dtype,
                       'ome_suffix':ome_suffix,
                       'cytosol_suffix':cytosol_suffix,
                       'nucleus_suffix':nucleus_suffix,
                       'filter_nuc_suffix':filter_nuc_suffix,
                       'filter_cel_suffix':filter_cel_suffix,
                       'separator':separator}


# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}_nef_translocation_hyperparameters_part2.csv"
hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name))

